# 03 - Machine Learning Model Training & Evaluation

**Project:** Customer Churn Intelligence Platform
**Dataset:** IBM Telco Customer Churn

This notebook performs stratified train/test split, trains and compares 4 classification models (Logistic Regression, Decision Tree, Random Forest, XGBoost), extracts feature importance, and serializes the champion pipeline.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(project_root))

from src.models.train import train_and_compare_models
from src.models.predict import ChurnPredictor

## 1. Execute Model Training & Benchmarking

In [ ]:
metadata = train_and_compare_models(
    data_path=str(project_root / 'dataset/processed/customers_clean.csv'),
    artifacts_dir=str(project_root / 'artifacts'),
    reports_dir=str(project_root / 'reports')
)
print(f'Champion Model: {metadata["model_name"]}')

## 2. Model Comparison Metrics

In [ ]:
pd.DataFrame(metadata['all_model_comparison']).T[['accuracy', 'precision', 'recall', 'f1_score', 'roc_auc', 'pr_auc']]

## 3. Top Predictive Features

In [ ]:
pd.Series(metadata['features']['top_10_features'], name='Importance').to_frame()

## 4. Test Inference with Saved Pipeline

In [ ]:
predictor = ChurnPredictor(
    model_path=str(project_root / 'artifacts/churn_pipeline.joblib'),
    metadata_path=str(project_root / 'artifacts/model_metadata.json')
)
sample_customer = {
    'gender': 'Female',
    'SeniorCitizen': 0,
    'Partner': 'No',
    'Dependents': 'No',
    'tenure': 2,
    'PhoneService': 'Yes',
    'MultipleLines': 'No',
    'InternetService': 'Fiber optic',
    'OnlineSecurity': 'No',
    'OnlineBackup': 'No',
    'DeviceProtection': 'No',
    'TechSupport': 'No',
    'StreamingTV': 'Yes',
    'StreamingMovies': 'No',
    'Contract': 'Month-to-month',
    'PaperlessBilling': 'Yes',
    'PaymentMethod': 'Electronic check',
    'MonthlyCharges': 79.85,
    'TotalCharges': 159.70
}
res = predictor.predict_single(sample_customer)
print('Sample Prediction Result:')
for k, v in res.items():
    print(f'{k:20s}: {v}')